# Module 1 — Exploratory Data Analysis (EDA)

**Project:** Fake Job Detection using Machine Learning

This notebook performs the first project module: dataset inspection, data-quality analysis, target/class distribution, numerical and categorical analysis, text-length analysis, and basic visualizations. Outputs are saved under `reports/EDA/` for use in the project report and PPT.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_DIR = Path('../Data/raw')
OUT_DIR = Path('../reports/EDA')
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RAW_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV dataset found in {RAW_DIR.resolve()}')

DATASET_PATH = csv_files[0]
print(f'Dataset selected: {DATASET_PATH}')


In [ ]:
df = pd.read_csv(DATASET_PATH)
print('Dataset loaded successfully.')
print('Shape:', df.shape)
df.head()

## 1. Dataset Structure

In [ ]:
structure = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null': df.notna().sum().values,
    'missing': df.isna().sum().values,
    'missing_%': (df.isna().mean().values * 100).round(2),
    'unique': df.nunique(dropna=True).values
})
display(structure)
structure.to_csv(OUT_DIR / 'dataset_structure.csv', index=False)

print('\nDuplicate rows:', df.duplicated().sum())
print('Rows:', len(df))
print('Columns:', len(df.columns))

## 2. Missing-Value Analysis

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print('No missing values found.')
else:
    missing_table = pd.DataFrame({
        'missing_count': missing,
        'missing_%': (missing / len(df) * 100).round(2)
    })
    display(missing_table)
    missing_table.to_csv(OUT_DIR / 'missing_values.csv')

    plt.figure(figsize=(10, 5))
    missing_table['missing_%'].sort_values(ascending=True).plot(kind='barh')
    plt.xlabel('Missing Values (%)')
    plt.ylabel('Column')
    plt.title('Missing Values by Column')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'missing_values.png', dpi=200)
    plt.show()

## 3. Target Variable Analysis

In [ ]:
target_candidates = ['fraudulent', 'fraud', 'is_fraud', 'label', 'target', 'class']
target_col = next((c for c in target_candidates if c in df.columns), None)

if target_col is None:
    print('Target column was not automatically identified.')
    print('Available columns:', list(df.columns))
else:
    print('Target column:', target_col)
    target_counts = df[target_col].value_counts(dropna=False)
    target_pct = (target_counts / len(df) * 100).round(2)
    target_table = pd.DataFrame({'count': target_counts, 'percentage': target_pct})
    display(target_table)
    target_table.to_csv(OUT_DIR / 'target_distribution.csv')

    plt.figure(figsize=(7, 5))
    target_counts.plot(kind='bar')
    plt.xlabel('Class')
    plt.ylabel('Number of Job Postings')
    plt.title('Target Class Distribution')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'target_distribution.png', dpi=200)
    plt.show()

## 4. Numerical Feature Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    numeric_summary = df[numeric_cols].describe().T
    display(numeric_summary)
    numeric_summary.to_csv(OUT_DIR / 'numerical_summary.csv')
else:
    print('No numerical columns detected.')


## 5. Categorical Feature Analysis

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
cat_rows = []
for col in categorical_cols:
    cat_rows.append({
        'column': col,
        'unique_values': df[col].nunique(dropna=True),
        'top_value': df[col].mode(dropna=True).iloc[0] if not df[col].mode(dropna=True).empty else np.nan,
        'top_count': int(df[col].value_counts(dropna=True).iloc[0]) if not df[col].value_counts(dropna=True).empty else 0
    })

categorical_summary = pd.DataFrame(cat_rows)
display(categorical_summary)
categorical_summary.to_csv(OUT_DIR / 'categorical_summary.csv', index=False)


## 6. Text Feature Analysis

In [ ]:
text_candidates = [
    c for c in ['title', 'company_profile', 'description', 'requirements', 'benefits']
    if c in df.columns
]

if text_candidates:
    text_summary = []
    for col in text_candidates:
        lengths = df[col].fillna('').astype(str).str.len()
        words = df[col].fillna('').astype(str).str.split().str.len()
        text_summary.append({
            'column': col,
            'avg_characters': round(lengths.mean(), 2),
            'median_characters': round(lengths.median(), 2),
            'avg_words': round(words.mean(), 2),
            'empty_records': int((lengths == 0).sum())
        })

    text_summary = pd.DataFrame(text_summary)
    display(text_summary)
    text_summary.to_csv(OUT_DIR / 'text_summary.csv', index=False)

    for col in text_candidates:
        lengths = df[col].fillna('').astype(str).str.len()
        plt.figure(figsize=(8, 5))
        plt.hist(lengths, bins=30)
        plt.xlabel('Character Count')
        plt.ylabel('Number of Job Postings')
        plt.title(f'Text Length Distribution: {col}')
        plt.tight_layout()
        plt.savefig(FIG_DIR / f'text_length_{col}.png', dpi=200)
        plt.show()
else:
    print('No standard text columns detected.')

## 7. Key Categorical Distributions

In [ ]:
plot_candidates = [
    c for c in ['employment_type', 'required_experience', 'required_education', 'industry', 'function']
    if c in df.columns
]

for col in plot_candidates:
    counts = df[col].fillna('Missing').value_counts().head(10).sort_values()
    plt.figure(figsize=(9, 5))
    counts.plot(kind='barh')
    plt.xlabel('Number of Job Postings')
    plt.ylabel(col)
    plt.title(f'Top Categories: {col}')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{col}_distribution.png', dpi=200)
    plt.show()

## 8. Correlation Analysis

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    display(corr)
    corr.to_csv(OUT_DIR / 'correlation_matrix.csv')

    plt.figure(figsize=(9, 7))
    plt.imshow(corr, aspect='auto')
    plt.colorbar(label='Correlation')
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title('Numerical Feature Correlation Matrix')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'correlation_matrix.png', dpi=200)
    plt.show()
else:
    print('Not enough numerical columns for correlation analysis.')

## 9. EDA Summary

In [ ]:
summary_lines = [
    '# Module 1 — EDA Summary',
    '',
    f'- Dataset: `{DATASET_PATH}`',
    f'- Rows: **{df.shape[0]}**',
    f'- Columns: **{df.shape[1]}**',
    f'- Duplicate rows: **{df.duplicated().sum()}**',
    f'- Columns with missing values: **{int((df.isna().sum() > 0).sum())}**',
]
if target_col is not None:
    summary_lines.append(f'- Target column: **{target_col}**')
    summary_lines.append('- Target distribution is available in `target_distribution.csv`.')

summary_lines += [
    '',
    'The detailed tables and figures generated by this notebook are stored in `reports/EDA/`.',
    'These outputs can be used for the EDA section of the project report and Review 1 PPT.'
]

(OUT_DIR / 'EDA_SUMMARY.md').write_text('\n'.join(summary_lines), encoding='utf-8')
print('\n'.join(summary_lines))